# BidBridge: Key Findings Walkthrough

This notebook walks through the key findings from the BidBridge research project, which studies the role of primary dealers as intermediaries in U.S. Treasury auctions.

In [ ]:
# Environment setup
# Requires: ~/venvs/bidbridge (Python 3.11)
# Install kernel if needed: pip install jupyter ipykernel
# Then: python -m ipykernel install --user --name bidbridge --display-name "BidBridge"

%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})

In [ ]:
# Project paths
ROOT = Path.cwd().parent
DATA = ROOT / 'data' / 'processed'
FIGS = ROOT / 'outputs' / 'figures'
TBLS = ROOT / 'outputs' / 'tables'

---
## 1. Overview

**BidBridge** investigates how primary dealers serve as a "bridge" between Treasury supply shocks and the broader fixed-income market. When auction supply surges, dealers absorb the excess onto their balance sheets, creating measurable inventory buildups. The project uses 846 weeks of auction, dealer-positioning, SOMA, and bank-holdings data (2010--2026) to quantify this mechanism and identify when it becomes strained.

In [ ]:
panel = pd.read_csv(DATA / 'auction_week_panel.csv', parse_dates=['week_start'])
print(f'Panel shape: {panel.shape[0]} weeks x {panel.shape[1]} columns')
print(f'Columns: {list(panel.columns)}')

---
## 2. Data Coverage

The panel merges five data sources at the weekly level. Not every source is available for every week, so coverage varies.

In [ ]:
date_min = panel['week_start'].min().strftime('%Y-%m-%d')
date_max = panel['week_start'].max().strftime('%Y-%m-%d')
print(f'Date range: {date_min}  to  {date_max}')
print(f'Total weeks: {len(panel)}')

In [ ]:
coverage = pd.DataFrame({
    'Source': ['Auctions (FiscalData)', 'PD positions (NY Fed)',
              'SOMA holdings (NY Fed)', 'H.8 bank holdings (FRED)',
              'Investor class (Treasury)'],
    'Key column': ['awarded_amount_total', 'pd_treasury_inventory',
                   'soma_treasury_total', 'bank_treasury_securities',
                   'dealer_share_allotment'],
})
coverage['Non-null weeks'] = coverage['Key column'].apply(
    lambda c: int(panel[c].notna().sum()))
coverage['Coverage %'] = (coverage['Non-null weeks'] / len(panel) * 100).round(1)
coverage

---
## 3. Supply Trends

Weekly auction supply has grown dramatically over the sample period, from roughly \$6 trillion per year to over \$30 trillion per year.

In [ ]:
fig, ax = plt.subplots()
ax.plot(panel['week_start'], panel['awarded_amount_total'] / 1e9,
        linewidth=0.7, color='steelblue')
ax.set_ylabel('Awarded amount ($B)')
ax.set_xlabel('Week')
ax.set_title('Weekly Treasury Auction Supply')
ax.axhline(panel['awarded_amount_total'].median() / 1e9,
           ls='--', color='grey', lw=0.8, label='median')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
annual = panel.set_index('week_start').resample('YE')['awarded_amount_total'].sum() / 1e12
print('Annual auction supply ($T):')
print(annual.to_string(float_format='{:.1f}'.format))

---
## 4. Dealer Absorption

A key finding is that dealer share of auction allotments is **negatively** correlated with supply (r = -0.82). This inverts the naive hypothesis that dealers would take more when there is more to buy. Instead, as supply has surged, end investors have absorbed a growing proportion, and dealers have become a **residual** buffer rather than the primary buyer.

The pre-generated scatter plot is at `outputs/figures/dealer_share_vs_supply.png`.

In [ ]:
display(Image(filename=str(FIGS / 'dealer_share_vs_supply.png'), width=700))

In [ ]:
mask = panel['dealer_share_allotment'].notna() & panel['awarded_amount_total'].notna()
r = panel.loc[mask, 'dealer_share_allotment'].corr(
    panel.loc[mask, 'awarded_amount_total'])
print(f'Pearson correlation (supply vs dealer share): r = {r:.2f}')

---
## 5. Bridge Episodes

A **bridge episode** is defined as a week where the dealer inventory change z-score exceeds +1: dealers absorbed significantly more inventory than their historical norm.

The panel flags 66 such episodes across the sample (roughly 7.8% of weeks). The table below summarizes episodes by year.

In [ ]:
n_episodes = int(panel['bridge_episode'].sum())
print(f'Total bridge episodes: {n_episodes}')
print(f'Share of sample: {n_episodes / len(panel) * 100:.1f}%')

In [ ]:
# Timeline of bridge episodes
fig, ax = plt.subplots(figsize=(14, 3))
ep = panel[panel['bridge_episode']]
ax.vlines(ep['week_start'], 0, 1, color='crimson', alpha=0.6, linewidth=1.5)
ax.set_yticks([])
ax.set_xlabel('Week')
ax.set_title(f'Bridge Episode Timeline ({n_episodes} episodes)')
plt.tight_layout()
plt.show()

In [ ]:
episode_summary = pd.read_csv(TBLS / 'bridge_episode_summary.csv')
episode_summary

Episodes have become larger over time. Average inventory change during episodes grew from ~\$18B in 2014 to ~\$53B in 2026, mirroring the growth in auction supply.

---
## 6. Event Studies

The pre-generated event study figures show dealer inventory dynamics around **refunding weeks** and **bridge episodes**.

In [ ]:
print('Refunding Week Event Study:')
display(Image(filename=str(FIGS / 'event_study_refunding.png'), width=700))

In [ ]:
print('Bridge Episode Event Study:')
display(Image(filename=str(FIGS / 'event_study_bridge_episodes.png'), width=700))

Refunding weeks see an average inventory buildup of ~\$10.4B compared to ordinary weeks (-\$1.8B). The Welch t-test confirms this is highly significant (t = 6.32, p < 0.001).

---
## 7. Maturity Structure

Dealer absorption varies sharply by maturity bucket. Bills see the highest dealer share (~62%), while TIPS see the lowest (~24%). This gradient reflects the fact that shorter-duration instruments are easier to warehouse and finance.

In [ ]:
mat = pd.read_csv(DATA / 'maturity_bucket_panel.csv', parse_dates=['week_start'])
print(f'Maturity panel: {mat.shape[0]} rows x {mat.shape[1]} columns')
print(f'Buckets: {sorted(mat["maturity_bucket"].unique())}')

In [ ]:
bucket_summary = pd.read_csv(TBLS / 'maturity_bucket_summary.csv')
cols = ['maturity_bucket', 'dealer_share_mean', 'dealer_share_std',
        'weighted_bid_to_cover_mean', 'bucket_share_of_weekly_mean']
bucket_summary[cols].sort_values('dealer_share_mean', ascending=False)

In [ ]:
display(Image(filename=str(FIGS / 'dealer_share_by_instrument.png'), width=700))

The gradient from bills (62%) through short coupons (38%), belly coupons (30%), long coupons (27%), and TIPS (24%) is consistent with an inventory-cost explanation: dealer willingness to absorb falls as duration and risk increase.

---
## 8. Stress Regimes

Four stress flags identify macro environments that may condition the dealer-bridge mechanism:

- **QT period**: SOMA declining for 4+ consecutive weeks
- **TGA rebuild**: High bill issuance to rebuild the Treasury General Account
- **Weak bank absorption**: Commercial bank Treasury holdings declining 4+ weeks
- **Risk-off window**: Auction tails above the 90th percentile

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from bidbridge.features.stress_flags import add_stress_flags

panel_s = add_stress_flags(panel)
flags = ['qt_period', 'tga_rebuild', 'weak_bank_absorption', 'risk_off_window']
print('Stress flag activation counts:')
print(panel_s[flags].sum().to_string())

In [ ]:
# Bridge rate during QT vs non-QT
qt_bridge = panel_s.loc[panel_s['qt_period'], 'bridge_episode'].mean()
non_qt_bridge = panel_s.loc[~panel_s['qt_period'], 'bridge_episode'].mean()
print(f'Bridge rate during QT:     {qt_bridge:.1%}')
print(f'Bridge rate outside QT:    {non_qt_bridge:.1%}')
print(f'Ratio:                     {qt_bridge / non_qt_bridge:.1f}x')

In [ ]:
stress_summary = pd.read_csv(TBLS / 'stress_regime_summary.csv')
stress_summary

In [ ]:
display(Image(filename=str(FIGS / 'stress_bridge_conditioning.png'), width=700))

Bridge episodes are roughly **2x more frequent during QT** (13.6% vs 7.5%) and **TGA rebuild** (13.3% vs 6.3%) periods, confirming that reduced SOMA absorption and heavy bill supply strain the dealer bridge.

---
## 9. Regression Results

Two OLS regressions model weekly dealer inventory change ($M):
- **Basic**: supply, dealer share, refunding dummy
- **Extended**: adds SOMA change, bank holdings change, and a linear trend (HC1 robust SEs)

In [ ]:
reg_basic = pd.read_csv(TBLS / 'regression_basic.csv')
print('=== Basic Regression ===')
reg_basic

In [ ]:
reg_ext = pd.read_csv(TBLS / 'regression_extended.csv')
print('=== Extended Regression ===')
reg_ext

### Key coefficients (extended model, R-squared = 0.175):

| Variable | Coefficient | Interpretation |
|----------|-------------|----------------|
| `supply_M` | +0.062 | +\$62M inventory per \$1B additional supply |
| `d_soma_B` | -82.3 | -\$82M inventory per \$1B SOMA increase (absorption substitute) |
| `refunding_week` | +5,467 | +\$5.5B inventory during refunding weeks |
| `trend_years` | -3,352 | -\$3.4B/yr secular decline in dealer warehousing |
| `dealer_share` | -33,364 | Higher dealer share at auction is associated with *lower* inventory change |

The SOMA coefficient confirms the substitution story: when the Fed buys more, dealers need to absorb less. The negative trend captures the structural shift toward end-investor absorption.

In [ ]:
ref_test = pd.read_csv(TBLS / 'regression_refunding_test.csv')
print('=== Refunding Week Welch t-test ===')
ref_test

---
## 10. Summary

**Five key takeaways from BidBridge:**

1. **Dealers are residual absorbers, not primary buyers.** Dealer share of auction allotments is strongly negatively correlated with supply (r = -0.82), falling from ~65% to ~32% as annual supply grew from \$6T to \$30T+.

2. **Bridge episodes identify acute warehousing strain.** 66 weeks (7.8% of the sample) saw inventory z-scores above +1. These episodes cluster during refunding weeks and periods of macro stress.

3. **SOMA is the key substitute for dealer absorption.** Each \$1B increase in SOMA holdings reduces dealer inventory change by \$82M, confirming the Fed's role as an absorption backstop. During QT, bridge episodes are 2x more frequent.

4. **Maturity structure matters.** Dealer absorption ranges from 62% for bills down to 24% for TIPS, consistent with an inventory-cost channel: longer duration = more risk = less willingness to warehouse.

5. **The extended regression explains 17.5% of inventory variation.** Supply (+\$62M/\$1B), SOMA (-\$82M/\$1B), refunding weeks (+\$5.5B), and a secular trend (-\$3.4B/yr) are all significant predictors of dealer inventory changes.